# =======
# Initializion
# =======

In [81]:
import pandas as pd
import numpy as np
import regex as re
import os

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

import meteostat as ms
from datetime import datetime, timedelta
import helper_functions as f

# spark biblioteker fra labene
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import (
    col, lit, when, count, sum as spark_sum, avg,
    min as spark_min, max as spark_max, round as spark_round,
    desc, asc, to_timestamp, to_date, hour, dayofweek,
    lower, upper, trim, concat, coalesce, explode, split,
    countDistinct, collect_list, first, input_file_name,
    regexp_extract, min, max
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, BooleanType, LongType
)

In [ ]:
# Create the SparkSession - the entry point for all Spark functionality
spark = SparkSession.builder \
    .appName("DAT535-Project-FootballStats") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

# ========
# Bronze layer
# ========

### OpenData Football data

In [73]:
LA_LIGA_MATCHES_PATH = "../open-data/data/matches/11"

matches_df = spark.read.json(LA_LIGA_MATCHES_PATH, multiLine=True).select(
    col("match_id"),
    col("match_date"),
    col("kick_off"),
    col("stadium.name").alias("stadium")
)

In [ ]:
EVENTS_PATH = "../open-data/data/events"

match_ids = matches_df.select(
    col("match_id")
).collect()

paths = []
for row in match_ids:
    paths.append(os.path.join(EVENTS_PATH, (str(row.match_id) + ".json")))

events_df = spark.read.json(paths, multiLine=True) \
                        .withColumn("match_id", 
                        regexp_extract(input_file_name(), r"([0-9]+).json", 1)) \
                        .select(
                            col("match_id"),
                            col("pass.height"),
                            col("pass.length"),
                            col("duration"),
                            col("pass.outcome")
                        )

events_df.write.mode("overwrite").parquet("bronze")

### Meteostat Weather data

In [ ]:
# Bundling up the first and last match for doing fewer, but bigget API-calls
unique_stadiums_df = matches_df.groupBy("stadium") \
                            .agg(min("match_date").alias("first_match"), max("match_date").alias("last_match"))


+--------------------+-----------+----------+
|             stadium|first_match|last_match|
+--------------------+-----------+----------+
|     Abanca-Balaídos| 2006-08-28|2020-10-01|
|            Bernabéu| 1974-02-17|2020-03-01|
|Campos de Sport d...| 2006-09-17|2012-03-11|
|Coliseum Alfonso ...| 2005-11-06|2020-10-17|
|Estadi Municipal ...| 2017-09-23|2019-01-27|
|Estadi Olímpic Ll...| 2004-10-16|2008-09-27|
|Estadio Abanca-Ri...| 2006-11-04|2018-04-29|
|Estadio Alfredo D...| 2021-04-10|2021-04-10|
|Estadio Benito Vi...| 2009-02-14|2021-02-07|
|Estadio Carlos Be...| 2004-12-11|2004-12-11|
|Estadio Ciudad de...| 2007-09-29|2021-05-11|
|  Estadio El Alcoraz| 2021-01-03|2021-01-03|
|    Estadio El Sadar| 2009-01-11|2021-03-06|
|Estadio Enrique R...| 2008-05-17|2008-05-17|
|Estadio Heliodoro...| 2010-01-10|2010-01-10|
|Estadio José Rico...| 2011-01-29|2011-01-29|
| Estadio La Rosaleda| 2008-11-01|2017-04-08|
|Estadio Manuel Ma...| 2014-05-11|2015-01-24|
|Estadio Municipal...| 2008-09-21|

In [97]:
from datetime import date
DATE = date(1976, 2, 17)

lat, long = f.get_coordinates("Bernabéu")
POINT = ms.Point(lat, long)
stations = ms.stations.nearby(POINT, limit=3)
time_series = ms.hourly(stations, DATE, DATE + timedelta(365)).fetch()

time_series

temp  rhum  prcp  snwd  wdir  wspd  wpgt    pres  \
station time                                                                    
08223   1976-02-17 06:00:00   2.0    75  <NA>  <NA>     0   0.0  <NA>  1017.1   
        1976-02-17 07:00:00   2.0    75  <NA>  <NA>   280   3.6  <NA>    <NA>   
        1976-02-17 08:00:00   2.0    80  <NA>  <NA>     0   0.0  <NA>    <NA>   
        1976-02-17 09:00:00   3.0    75  <NA>  <NA>   310   3.6  <NA>  1017.3   
        1976-02-17 10:00:00   4.0    70  <NA>  <NA>     0   0.0  <NA>    <NA>   
...                           ...   ...   ...   ...   ...   ...   ...     ...   
        1977-02-16 14:00:00   9.0    71  <NA>  <NA>   250  14.8  <NA>    <NA>   
        1977-02-16 15:00:00  12.0    50  <NA>  <NA>   250  25.9  <NA>  1021.6   
        1977-02-16 16:00:00  13.0    47  <NA>  <NA>   250  22.0  <NA>    <NA>   
        1977-02-16 17:00:00  12.0    54  <NA>  <NA>   250  18.4  <NA>    <NA>   
        1977-02-16 18:00:00  10.0    76  <NA>  <NA>   250  18.4  <NA>  1022.2   

                             tsun  cldc  coco  
station time                                   
08223   1976-02-17 06:00:00  <NA>     7  <NA>  
        1976-02-17 07:00:00  <NA>  <NA>  <NA>  
        1976-02-17 08:00:00  <NA>  <NA>  <NA>  
        1976-02-17 09:00:00  <NA>     6  <NA>  
        1976-02-17 10:00:00  <NA>  <NA>  <NA>  
...                           ...   ...   ...  
        1977-02-16 14:00:00  <NA>  <NA>  <NA>  
        1977-02-16 15:00:00  <NA>     7  <NA>  
        1977-02-16 16:00:00  <NA>  <NA>  <NA>  
        1977-02-16 17:00:00  <NA>  <NA>  <NA>  
        1977-02-16 18:00:00  <NA>     5  <NA>  

[4039 rows x 11 columns]

In [ ]:
from datetime import date
import time

ms.config.block_large_requests = False

start = time.perf_counter()

for row in unique_stadiums_df.collect():
    
    lat, long = f.get_coordinates(row.stadium)

    print(row.first_match, type(row.first_match))
    # Creates a datetime from a string (e.g. datetime.striptime("2026-11-03 17:07:01.000", "%Y-%m-%d %H:%M:%S.%f"))
    first_match = date.strptime(row.first_match, "%Y-%m-%d")
    last_match = date.strptime(row.last_match, "%Y-%m-%d")
    POINT = ms.Point(lat, long)
    stations = ms.stations.nearby(POINT, limit=4)

    time_series = ms.hourly(stations, first_match, last_match)
    weather = ms.interpolate(time_series, POINT).fetch()
    break

print(time.perf_counter() - start, "seconds")

2006-08-28 <class 'str'>


KeyboardInterrupt: 

# =======
# Silver layer
# =======

# =======
# Gold layer
# =======